# 📈 Ridge & Lasso Regression — Practice Notebook

**Difficulty**: ⭐⭐ Intermediate  
**Time**: ~50 minutes  
**Prerequisites**: Linear Regression, Polynomial Regression

---

## What You'll Learn
- L2 regularization (Ridge) — why and how
- L1 regularization (Lasso) — feature selection power
- Elastic Net — the best of both worlds
- Geometric intuition of L1 vs L2
- Choosing the regularization strength (α)
- Coefficient paths visualization

---
## 🎯 Section 1: Overview

**Regularization** adds a penalty term to the cost function to prevent overfitting by discouraging large coefficients.

### Ridge Regression (L2 Regularization)
$$J(\theta) = \text{MSE}(\theta) + \alpha \sum_{j=1}^{n} \theta_j^2$$

- Shrinks coefficients toward zero but **never exactly zero**
- Good when many features contribute a little
- Has a closed-form solution

### Lasso Regression (L1 Regularization)
$$J(\theta) = \text{MSE}(\theta) + \alpha \sum_{j=1}^{n} |\theta_j|$$

- Can shrink coefficients to **exactly zero** → automatic feature selection!
- Good when you suspect only a few features matter
- No closed-form solution (uses coordinate descent)

### Elastic Net (L1 + L2)
$$J(\theta) = \text{MSE}(\theta) + \alpha \left( r \sum |\theta_j| + \frac{1-r}{2} \sum \theta_j^2 \right)$$

where r is the L1 ratio (r=1 → Lasso, r=0 → Ridge)

### 🔑 Key Interview Insight: Why L1 Produces Sparsity
The L1 penalty creates a **diamond-shaped** constraint region. The optimal solution tends to hit the **corners** of the diamond, where some coefficients are exactly zero. L2 creates a **circular** constraint region, so the optimal solution rarely lands exactly on an axis.

---
## 📐 Section 2: Math & Intuition

### Ridge Closed-Form Solution
$$\theta = (X^T X + \alpha I)^{-1} X^T y$$

Note: The identity matrix I should have a 0 in the top-left (we don't regularize the bias θ₀).

**Key property**: Adding αI makes the matrix always invertible — this solves the multicollinearity problem!

### Gradient for Ridge
$$\nabla J = \frac{1}{m} X^T(X\theta - y) + \frac{2\alpha}{m} \theta$$

### Subgradient for Lasso
$$\nabla J = \frac{1}{m} X^T(X\theta - y) + \frac{\alpha}{m} \text{sign}(\theta)$$

### Regularization Strength (α)
| α value | Effect |
|---|---|
| α = 0 | No regularization (plain linear regression) |
| Small α | Mild regularization, coefficients slightly shrunk |
| Large α | Strong regularization, coefficients → 0 |
| α → ∞ | All coefficients = 0 (underfitting) |

---
## 🔧 Section 3: Implementation from Scratch

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print('Setup complete! ✅')

### 3.1 Generate Data with Many Features (Some Irrelevant)

In [ ]:
# Dataset: 3 relevant features, 7 irrelevant features
np.random.seed(42)
m = 200
n_features = 10

X = np.random.randn(m, n_features)
# True relationship uses only features 0, 1, 2
true_theta = np.array([5.0, -3.0, 2.0, 0, 0, 0, 0, 0, 0, 0])
y = X @ true_theta + 3.0 + np.random.randn(m) * 0.5  # intercept=3, noise=0.5

print(f'X shape: {X.shape}')
print(f'True coefficients: {true_theta}')
print(f'Only features 0, 1, 2 are relevant!')

# Train/test split
split = int(0.8 * m)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

### 3.2 Ridge Regression from Scratch

In [ ]:
def ridge_regression(X, y, alpha):
    """
    Ridge regression using closed-form solution.
    θ = (X^T X + αI)^(-1) X^T y
    
    Parameters:
    -----------
    X     : np.ndarray of shape (m, n) — features (WITHOUT bias column)
    y     : np.ndarray of shape (m,)   — target
    alpha : float — regularization strength
    
    Returns:
    --------
    theta     : np.ndarray of shape (n,) — coefficients (no intercept)
    intercept : float — bias term
    """
    # TODO: Implement Ridge regression
    # Step 1: Center the data (subtract means) to handle intercept separately
    #   X_centered = X - X.mean(axis=0)
    #   y_centered = y - y.mean()
    # Step 2: Compute theta = (X_c^T X_c + alpha*I)^(-1) X_c^T y_c
    # Step 3: Compute intercept = y.mean() - X.mean(axis=0) @ theta
    
    theta = None      # TODO: Replace
    intercept = None   # TODO: Replace
    
    return theta, intercept


# Test
theta_ridge, intercept_ridge = ridge_regression(X_train, y_train, alpha=1.0)
assert theta_ridge is not None, "Implement ridge_regression!"

print(f'Ridge (α=1.0) coefficients:')
for i, (t, true_t) in enumerate(zip(theta_ridge, true_theta)):
    print(f'  θ{i} = {t:7.4f}  (true: {true_t:.1f})')
print(f'  intercept = {intercept_ridge:.4f}  (true: 3.0)')

### 3.3 Lasso Regression from Scratch (Coordinate Descent)

In [ ]:
def soft_threshold(rho, alpha):
    """Soft thresholding operator for Lasso."""
    if rho > alpha:
        return rho - alpha
    elif rho < -alpha:
        return rho + alpha
    else:
        return 0.0


def lasso_coordinate_descent(X, y, alpha, num_iters=1000, tol=1e-6):
    """
    Lasso regression using coordinate descent.
    
    Parameters:
    -----------
    X         : np.ndarray of shape (m, n)
    y         : np.ndarray of shape (m,)
    alpha     : float — L1 regularization strength
    num_iters : int — max iterations
    tol       : float — convergence tolerance
    
    Returns:
    --------
    theta     : np.ndarray of shape (n,)
    intercept : float
    """
    m, n = X.shape
    
    # Center data
    X_mean = X.mean(axis=0)
    y_mean = y.mean()
    X_c = X - X_mean
    y_c = y - y_mean
    
    theta = np.zeros(n)
    
    # TODO: Implement coordinate descent
    # For each iteration:
    #   For each feature j:
    #     1. Compute residual WITHOUT feature j: r_j = y_c - X_c @ theta + X_c[:, j] * theta[j]
    #     2. Compute rho_j = X_c[:, j] @ r_j / m
    #     3. Compute z_j = X_c[:, j] @ X_c[:, j] / m
    #     4. Update: theta[j] = soft_threshold(rho_j, alpha) / z_j
    #   Check convergence: if max change in theta < tol, break
    
    # YOUR CODE HERE
    pass
    
    intercept = y_mean - X_mean @ theta
    return theta, intercept


# Test
theta_lasso, intercept_lasso = lasso_coordinate_descent(X_train, y_train, alpha=0.1)

print(f'Lasso (α=0.1) coefficients:')
for i, (t, true_t) in enumerate(zip(theta_lasso, true_theta)):
    zero_marker = ' ← ZEROED OUT' if abs(t) < 1e-6 else ''
    print(f'  θ{i} = {t:7.4f}  (true: {true_t:.1f}){zero_marker}')
print(f'  intercept = {intercept_lasso:.4f}  (true: 3.0)')
print(f'\n# Zero coefficients: {sum(abs(t) < 1e-6 for t in theta_lasso)}/10')

---
## 📦 Section 4: Using scikit-learn

In [ ]:
from sklearn.linear_model import Ridge, Lasso, ElasticNet, RidgeCV, LassoCV, ElasticNetCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

# TODO: Compare Ridge, Lasso, and ElasticNet using sklearn
# Step 1: Scale the features using StandardScaler
# Step 2: Fit Ridge(alpha=1.0), Lasso(alpha=0.1), ElasticNet(alpha=0.1, l1_ratio=0.5)
# Step 3: For each model, print:
#   - Coefficients
#   - Number of zero coefficients
#   - Test R²
#   - Test RMSE
# Step 4: Compare — which model identifies the true relevant features?

# YOUR CODE HERE


### 4.1 Automatic Alpha Selection with Cross-Validation

In [ ]:
# TODO: Use RidgeCV and LassoCV to automatically find the best alpha
# RidgeCV: try alphas = np.logspace(-4, 4, 50)
# LassoCV: uses built-in CV path

# Print the best alpha for each and compare R² scores

# YOUR CODE HERE


---
## 🧪 Section 5: Experiments

### 5.1 Coefficient Paths (Regularization Path)

In [ ]:
# TODO: Plot coefficient paths for Ridge and Lasso
# For alphas = np.logspace(-2, 4, 100):
#   Fit Ridge/Lasso and record all coefficients
# Plot: alpha (x-axis, log scale) vs coefficient value (y-axis)
# Each feature gets its own line
# Create a 1x2 subplot: Ridge on left, Lasso on right
#
# Observation: In Lasso, coefficients hit ZERO as alpha increases.
#              In Ridge, coefficients approach zero but never reach it.

# YOUR CODE HERE


### 5.2 Ridge vs Lasso on High-Degree Polynomial

In [ ]:
# Generate nonlinear data
np.random.seed(42)
X_nl = np.sort(np.random.uniform(-3, 3, 50)).reshape(-1, 1)
y_nl = 1 + 0.5 * X_nl.ravel() + 2 * X_nl.ravel()**2 + np.random.randn(50) * 2

# TODO: Show how Ridge/Lasso tame a degree-15 polynomial
# 1. Create degree-15 polynomial features (use PolynomialFeatures)
# 2. Scale the features
# 3. Fit: plain LinearRegression, Ridge(alpha=1), Lasso(alpha=1)
# 4. Plot all 3 fits on the same graph
# 5. Observe: unregularized goes wild, Ridge/Lasso are smooth

# Hint: from sklearn.preprocessing import PolynomialFeatures

# YOUR CODE HERE


### 5.3 Multicollinearity Handling

In [ ]:
# TODO: Demonstrate how Ridge handles multicollinearity
# 1. Create correlated features: x2 = x1 + small_noise
# 2. Fit ordinary linear regression — observe unstable coefficients
# 3. Fit Ridge regression — observe stable coefficients
# 4. Print and compare coefficients

# Hint:
# X_collinear = np.column_stack([x1, x1 + np.random.randn(m)*0.01, x3])

# YOUR CODE HERE


---
## ❓ Section 6: Interview Questions

### Q1: When should you use Ridge vs Lasso?
<details><summary>Click for answer</summary>

- **Ridge**: When you believe many features contribute small amounts. Ridge keeps all features but shrinks them.
- **Lasso**: When you suspect only a few features are truly relevant. Lasso performs automatic feature selection.
- **Elastic Net**: When features are correlated. Lasso might arbitrarily pick one of correlated features; Elastic Net handles this better.

Rule of thumb: Start with Ridge if you're unsure. Use Lasso if you want interpretability/sparsity.
</details>

### Q2: Why does L1 (Lasso) produce sparse solutions but L2 (Ridge) doesn't?
<details><summary>Click for answer</summary>

**Geometric intuition**: The constraint region for L1 is a **diamond** (has corners on axes). The constraint region for L2 is a **circle** (smooth, no corners). The optimal solution is where the MSE contours first touch the constraint region. The diamond's corners sit on axes where some θ=0, making sparse solutions likely. The circle has no corners, so the solution rarely lands exactly on an axis.

**Mathematical intuition**: The L1 subdifferential at θ=0 is the interval [-1, 1], so there's a range of gradient values for which θ stays at 0. For L2, the gradient at θ=0 is exactly 0, which only happens when the MSE gradient is also 0 — much rarer.
</details>

### Q3: How do you choose the regularization parameter α?
<details><summary>Click for answer</summary>

- **Cross-validation** (most common) — RidgeCV, LassoCV in sklearn
- **Grid search** over a range: typically α ∈ [10⁻⁴, 10⁴]
- **Information criteria** — some implementations use AIC/BIC
- **Regularization path** — plot coefficients vs α, look for stability
</details>

### Q4: What is Elastic Net and when would you use it?
<details><summary>Click for answer</summary>

Elastic Net combines L1 and L2 penalties. Use it when:
- Features are **correlated** (Lasso struggles, may pick one arbitrarily)
- You want **some sparsity** but also stability
- Number of features > number of samples (Lasso can select at most n features)

It has two hyperparameters: α (overall strength) and l1_ratio (mix of L1/L2).
</details>

### Q5: Should you standardize features before regularization? Why?
<details><summary>Click for answer</summary>

**YES, always!** Regularization penalizes large coefficients. If features are on different scales, features with larger scales will have smaller coefficients and be penalized less, regardless of their importance. Standardizing ensures all features are penalized equally.

Note: scikit-learn's Ridge/Lasso do NOT automatically standardize — you must do it yourself (or use a Pipeline with StandardScaler).
</details>

### Q6: How does Ridge solve the multicollinearity problem?
<details><summary>Click for answer</summary>

In OLS, the solution requires inverting X^T X. When features are collinear, X^T X is nearly singular (det ≈ 0), making the inversion numerically unstable and coefficients unreliable.

Ridge adds αI to X^T X: `(X^T X + αI)^(-1)`. This makes the matrix always positive definite and invertible, stabilizing the solution. The eigenvalues of X^T X + αI are at least α, preventing near-zero denominators.
</details>

---
## 🏆 Section 7: Challenge

### Challenge: Feature Selection on a Real-ish Dataset

In [ ]:
# Synthetic dataset: predict exam score from study habits
np.random.seed(42)
n = 300

# 5 relevant features
hours_studied = np.random.uniform(1, 10, n)
practice_tests = np.random.randint(0, 20, n).astype(float)
sleep_hours = np.random.uniform(4, 10, n)
attendance_pct = np.random.uniform(50, 100, n)
prev_gpa = np.random.uniform(2.0, 4.0, n)

# 10 irrelevant (noise) features
noise_features = np.random.randn(n, 10)

# True score = f(relevant features) + noise
exam_score = (5 * hours_studied + 2 * practice_tests + 3 * sleep_hours 
              + 0.5 * attendance_pct + 10 * prev_gpa 
              + np.random.randn(n) * 3)

# Combine all features
feature_names = (['hours_studied', 'practice_tests', 'sleep_hours', 
                  'attendance_pct', 'prev_gpa'] + 
                 [f'noise_{i}' for i in range(10)])
X_exam = np.column_stack([hours_studied, practice_tests, sleep_hours, 
                          attendance_pct, prev_gpa, noise_features])

print(f'Features: {X_exam.shape[1]} (5 relevant + 10 noise)')
print(f'Samples: {n}')

In [ ]:
# TODO: Complete the challenge
# 1. Split into train/test (80/20)
# 2. Standardize features
# 3. Fit: LinearRegression, RidgeCV, LassoCV, ElasticNetCV
# 4. For each model, create a bar chart of coefficient magnitudes
# 5. Which model correctly identifies the 5 relevant features?
# 6. Compare test R² and RMSE across all models
# 7. Print a summary table

# YOUR CODE HERE


---
## ✅ Summary

- [x] Ridge (L2) shrinks coefficients but never zeroes them
- [x] Lasso (L1) performs automatic feature selection
- [x] Elastic Net combines both — great for correlated features
- [x] Always standardize before regularization
- [x] Use CV to choose regularization strength
- [x] Geometric intuition: diamond (L1) vs circle (L2)

**This completes Phase 1!** 🎉  
**Next Phase**: [Classification](../../classification/) →